In [79]:
import pandas as pd
import numpy as np

from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction, DefaultEmbeddingFunction

import torch
from transformers import pipeline

# Step 1 — Convert match records into text sentences

In [2]:
df = pd.read_csv('matches.csv')
df1 = pd.read_csv('matches_25.csv')
df2 = pd.read_csv('match_data2.csv')

In [45]:
corpus = []

template = "{team1} vs {team2} at {venue}, {city} on {date} ({season} IPL, {match_type}). {toss_winner} won the toss and chose to {toss_decision}. {winner} won by {result_margin} {result}, chasing a target of {target_runs} in {target_overs} overs. Player of the Match: {player_of_match}. Umpires: {umpire1} and {umpire2}."
template1 = "{team1} vs {team2} at {venue} on {date}. {match_winner} won. Player of the Match: {player_of_the_match}."
template2 = "{team1} vs {team2} at {venue} on {date}. {winning_team} won. Player of the Match: {player_of_match} (impact score: {player_of_match_total_impact}). Top scorer: {top_scorer} with {top_scorer_runs} runs."

print('1st Dataset(2008-2024) \n')
for idx,row in df.iterrows():

    corpus.append(
        f"{row['team1']} vs {row['team2']} at {row['venue']}, {row['city']} on {row['date']} ({row['season']} IPL, {row['match_type']}). {row['toss_winner']} won the toss and chose to {row['toss_decision']}. {row['winner']} won by {row['result_margin']} {row['result']}, chasing a target of {row['target_runs']} in {row['target_overs']} overs. Player of the Match: {row['player_of_match']}. Umpires: {row['umpire1']} and {row['umpire2']}."
    )
    print(idx, end=': ')

print('\n\n2nd Dataset(2025) \n')
for idx,row in df1.iterrows():

    corpus.append(
        f"{row['stage']} match {row['team1']} vs {row['team2']} at {row['venue']} on {row['date']}. {row['match_winner']} won by margin of {row['margin']}. Player of the Match: {row['player_of_the_match']}."
    )
    print(idx, end=': ')

print('\n\n3rd Dataset(2026) \n')
for idx,row in df2.iterrows():

    corpus.append(
        f"{row['Match_no']} {row['Team1']} vs {row['Team2']} at {row['Venue']} on {row['Date']}. {row['Winning_team']} won. Player of the Match: {row['Player_of_match']} (impact score: {row['Player_of_match_total_impact']}). Top scorer: {row['Top_scorer']} with {row['Top_scorer_runs']} runs."
    )
    print(idx, end=': ')


1st Dataset(2008-2024) 

0: 1: 2: 3: 4: 5: 6: 7: 8: 9: 10: 11: 12: 13: 14: 15: 16: 17: 18: 19: 20: 21: 22: 23: 24: 25: 26: 27: 28: 29: 30: 31: 32: 33: 34: 35: 36: 37: 38: 39: 40: 41: 42: 43: 44: 45: 46: 47: 48: 49: 50: 51: 52: 53: 54: 55: 56: 57: 58: 59: 60: 61: 62: 63: 64: 65: 66: 67: 68: 69: 70: 71: 72: 73: 74: 75: 76: 77: 78: 79: 80: 81: 82: 83: 84: 85: 86: 87: 88: 89: 90: 91: 92: 93: 94: 95: 96: 97: 98: 99: 100: 101: 102: 103: 104: 105: 106: 107: 108: 109: 110: 111: 112: 113: 114: 115: 116: 117: 118: 119: 120: 121: 122: 123: 124: 125: 126: 127: 128: 129: 130: 131: 132: 133: 134: 135: 136: 137: 138: 139: 140: 141: 142: 143: 144: 145: 146: 147: 148: 149: 150: 151: 152: 153: 154: 155: 156: 157: 158: 159: 160: 161: 162: 163: 164: 165: 166: 167: 168: 169: 170: 171: 172: 173: 174: 175: 176: 177: 178: 179: 180: 181: 182: 183: 184: 185: 186: 187: 188: 189: 190: 191: 192: 193: 194: 195: 196: 197: 198: 199: 200: 201: 202: 203: 204: 205: 206: 207: 208: 209: 210: 211: 212: 213: 214: 215: 216: 

In [54]:
print(len(corpus), '=', (1095 + 74 + 74))

1243 = 1243


In [58]:
# 1. One record per line (each chunk = one match, zero overlap)
with open("corpus_per_match.txt", "w", encoding="utf-8") as f:
    for record in corpus:
        f.write(record + "\n")

# 2. Full text save
full_text = " ".join(corpus)
with open("corpus_full.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [59]:
len(full_text)

371100

# Step 2 — Generate vector embeddings

In [7]:
with open('corpus_per_match.txt', 'r') as f:
    l = f.readlines()

In [8]:
L = [i.replace('\n', '') for i in l]

In [9]:
docs = [Document(page_content=line, metadata={'index':i}) for i,line in enumerate(L)]

In [ ]:
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4943.21it/s]


In [61]:
embeddings = model.encode(L)

In [59]:
model.encode('hello')

array([ 0.05105216,  0.00719569,  0.00387392, ..., -0.03147078,
       -0.0355579 , -0.01076884], shape=(1024,), dtype=float32)

In [62]:
len(embeddings)

1243

# Step 3 — Store the vectors

In [4]:
sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda",
    normalize_embeddings=False
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4055.56it/s]


In [80]:
default_ef = DefaultEmbeddingFunction()

In [41]:
chroma_client = chromadb.PersistentClient()

In [81]:
chroma_client2 = chromadb.Client()

In [42]:
s = sentence_transformer_ef('hello')

In [43]:
s

[array([-0.01874905,  0.00039714,  0.00991078, ..., -0.03231281,
         0.01497433, -0.04961997], shape=(1024,), dtype=float32),
 array([-0.03422008, -0.01299287, -0.01136643, ..., -0.01189206,
        -0.00352982, -0.00484265], shape=(1024,), dtype=float32),
 array([ 0.01158388, -0.01905851,  0.03038345, ..., -0.01277656,
        -0.04970322,  0.01772168], shape=(1024,), dtype=float32),
 array([ 0.01158388, -0.01905851,  0.03038345, ..., -0.01277656,
        -0.04970322,  0.01772168], shape=(1024,), dtype=float32),
 array([-0.01196045, -0.04163621,  0.00072617, ..., -0.03330432,
        -0.03298532,  0.01360258], shape=(1024,), dtype=float32)]

In [45]:
collection = chroma_client.create_collection(
    name="ipl_matches_08_26_2",
    embedding_function=sentence_transformer_ef
)

In [ ]:
collection.add(
    ids=[str(i) for i in range(len(L))],
    documents=L
)

In [82]:
collection2 = chroma_client2.create_collection(
    name="ipl_matches",
    embedding_function=default_ef
)

In [84]:
collection2.add(
    ids=[str(i) for i in range(len(L))],
    documents=L
)

C:\Users\shiva\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:12<00:00, 6.87MiB/s]


In [85]:
res = collection2.query(
    query_texts=["ipl winner of 2025"],
    n_results=10
)

In [86]:
res

{'ids': [['868',
   '878',
   '872',
   '871',
   '875',
   '858',
   '865',
   '897',
   '850',
   '761']],
 'embeddings': None,
 'documents': [['Chennai Super Kings vs Punjab Kings at Dubai International Cricket Stadium, Dubai on 2021-10-07 (2021 IPL, League). Punjab Kings won the toss and chose to field. Punjab Kings won by 6.0 wickets, chasing a target of 135.0 in 20.0 overs. Player of the Match: KL Rahul. Umpires: K Srinivasan and RK Illingworth.',
   'Royal Challengers Bangalore vs Punjab Kings at Dr DY Patil Sports Academy, Mumbai, Mumbai on 2022-03-27 (2022 IPL, League). Punjab Kings won the toss and chose to field. Punjab Kings won by 5.0 wickets, chasing a target of 206.0 in 20.0 overs. Player of the Match: OF Smith. Umpires: Nitin Menon and YC Barde.',
   'Delhi Capitals vs Chennai Super Kings at Dubai International Cricket Stadium, Dubai on 2021-10-10 (2021 IPL, Qualifier 1). Chennai Super Kings won the toss and chose to field. Chennai Super Kings won by 4.0 wickets, chasin

# Step 4 — Build a semantic search function

In [ ]:
sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda",
    normalize_embeddings=False
)

In [63]:
sentence_transformer_ef(['give me the result of ipl match on March 22,2025.'])

[array([ 0.00742967, -0.01920805, -0.05374198, ..., -0.05992454,
        -0.03273422, -0.03924677], shape=(1024,), dtype=float32)]

In [49]:
chroma_client = chromadb.PersistentClient()

In [50]:
new = chroma_client.get_collection('ipl_matches_08_26_2', sentence_transformer_ef)

In [65]:
new.query(
    query_texts=['give me RCB final match at 2025'],
    n_results=3
)['documents']

[['Final match RCB vs PBKS at Narendra Modi Stadium, Ahmedabad on June 3,2025. RCB won by margin of 6 runs. Player of the Match: Krunal Pandya.',
  'League match RCB vs DC at M. Chinnaswamy Stadium, Bangalore on April 10,2025. DC won by margin of 6 wickets. Player of the Match: KL Rahul.',
  'League match RR vs RCB at Sawai Mansingh Stadium, Jaipur on April 13,2025. RCB won by margin of 9 wickets. Player of the Match: Phil Salt.']]

# Step 5 — Final

In [ ]:
sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda",
    normalize_embeddings=False
)

In [68]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

pipe = pipeline(
    "text-generation", 
    model=model_id, 
    dtype=torch.bfloat16, 
    device_map="auto"
)

c:\Users\shiva\anaconda3\envs\third_ds\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiva\.cache\huggingface\hub\models--meta-llama--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████

In [ ]:
chroma_client = chromadb.PersistentClient()
new = chroma_client.get_collection('ipl_matches_08_26_2', sentence_transformer_ef)

In [77]:
while True:
    query = input('(type e to end)Enter Query: ')
    if query == 'e':
        break

    res = new.query(query_texts=[query])['documents']

    messages = [
        {"role": "system", "content": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise."},
        {"role": "user", "content": res},
    ]

    outputs = pipe(messages, max_new_tokens=256)

    print(outputs[0]["generated_text"][-1])

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'The player of the match for Kolkata Knight Riders in the match played on 2024-05-26 (2024 IPL, Final) at MA Chidambaram Stadium, Chepauk, Chennai, was MA Starc.'}
